
# 03. A Vasúti Árhatás: Zajdiszkont, Elérhetőségi Prémium és Térbeli Függés

**MIT mér ez a fejezet?** Azt, hogy egy vasútvonal hogyan hat a környezetében meghirdetett lakások árára — méghozzá egyszerre két, egymással ellentétes irányban:

- **Negatív oldal (immisszió):** a nyílt vágány közvetlen közelében a zaj, a rezgés, a por és a vizuális szennyezés környezeti terhet jelent, ami **árcsökkentő diszkontként** épül be a hirdetési árakba.
- **Pozitív oldal (elérhetőség):** az állomások gyalogos vonzáskörzetében a gyors, dugómentes kapcsolat időt takarít meg, ez **elérhetőségi prémiumként** (TOD-hatás) növeli az árat.

Ugyanaz az infrastruktúra tehát az egyik helyen le-, a másikon felértékeli az ingatlant; a fejezet célja a két erő **szétválasztása és számszerűsítése**.

**MIÉRT erre a mérési láncra van szükség?**

1. **Nyers gradiens:** először sávokra osztjuk a mintát a vágánytól mért távolság szerint, és sávonként összehasonlítjuk a medián fajlagos árakat. Ez gyors és szemléletes, de a nyers mediánarány **összetételi torzítást** rejt: a sávokban más-más a lakások mérete, kora és állapota, ezért a nyers különbség nem tiszta vasúti hatás.
2. **Kontrollált (hedonikus) modell:** a nyers gradiens torzítását úgy szűrjük ki, hogy az árat egyetlen regresszióban, *a többi tulajdonság változatlansága mellett* magyarázzuk — így a sáv-együtthatók már a "tiszta" környezeti hatást mérik.
3. **Térbeli modellek:** az ingatlanárak térben nem függetlenek (a szomszédos árak együttmozognak), ezért Moran-teszttel kimutatjuk a térbeli autokorrelációt, SAR/SEM modellekkel korrigáljuk a becslést, végül GWR-rel megmutatjuk, hogy a vasúti hatás maga is változik térben.

**HOGYAN? A kanonikus specifikáció és a referencia-zóna elve.** A vasúti hatást *küszöbös* formában mérjük: az immissziót fix távolsági sávok (0–150–300–500–1000–2000 m) diszjunkt dummy-változóival, az elérhetőséget gyalogos izokrón-sávokkal (0–375 / 375–750 / 750–1125 m), a függő változó a fajlagos ár természetes logaritmusa, a becslés robusztus (HC1) standard hibákkal történik — így az együtthatók közvetlenül százalékos prémiumként / diszkontként olvashatók, és minden fejezetben ugyanazok a számok szerepelnek. A zóna-dummyk referenciája az **adaptív referencia-zóna**: a legtávolabbi sáv, amelyben még elegendő megfigyelés van — ha a legtávolabbi sáv üres vagy túl kicsi, a referencia automatikusan a legközelebbi kellő méretű sávra esik, így a modell soha nem degenerálódik, a diszkont pedig mindig az adott terület zajterhelés-mentesnek tekinthető zónájához viszonyított.

A notebook a `TDK_ACTIVE_AREA` környezeti változóval kiválasztott területen fut; a terület nevét az alábbi fejléc dinamikusan, az adatkonfigurációból (`get_area_metadata(...)['name']`) jeleníti meg. A fejezet objektíven, területfüggetlen módszertannal dolgozik: minden szám a megosztott elemző modul (`ingatlan_tdk.analyses`) gyorstárából érkezik, a notebook csak megjelenít.


In [ ]:

import warnings; warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, '..')
from ingatlan_tdk._utils import *
from ingatlan_tdk import analyses as A
setup_plotly()
df = load_szamitott_master()
p = A.pontos_elado()
AREA = os.environ.get('TDK_ACTIVE_AREA', 'kobanya')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

area_nev = get_area_metadata(AREA)['name']
print(f"Aktív terület: {area_nev}  |  pontos eladó minta: {len(p)} db")
display(HTML(render_area_header_html(df=df)))

# --- megjelenítési segédek (nincs számítás, csak formázás; az eredmények a gyorstárból jönnek) ---
def fnum(v, nd=3):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return '—'
    return f"{v:.{nd}f}".replace('.', ',')

def fpct(v, nd=1):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return '—'
    return f"{v:+.{nd}f}%".replace('.', ',')

_CS = "border:1px solid #e2e8f0; padding:4px 10px;"

def plain_table_html(d, highlight_col=None, highlight_val=None):
    h = "<tr>" + "".join(f"<th style='{_CS}background:#f1f5f9;'>{c}</th>" for c in d.columns) + "</tr>"
    rows = []
    for _, r in d.iterrows():
        hl = highlight_col is not None and str(r[highlight_col]) == str(highlight_val)
        bg = "background:#fef3c7;" if hl else ""
        rows.append("<tr>" + "".join(f"<td style='{_CS}{bg}'>{v}</td>" for v in r) + "</tr>")
    return f"<table style='border-collapse:collapse;font-size:12px;margin:6px 0 14px 0;'>{h}{''.join(rows)}</table>"

def coef_pct(t):
    out = t.copy()
    out['hatas_pct'] = (np.exp(out['coef']) - 1) * 100
    out['ci_lo_pct'] = (np.exp(out['ci_lo']) - 1) * 100
    out['ci_hi_pct'] = (np.exp(out['ci_hi']) - 1) * 100
    return out

def coef_table_html(t, highlight=()):
    h = (f"<tr><th style='{_CS}background:#f1f5f9;'>Változó</th>"
         f"<th style='{_CS}background:#f1f5f9;'>coef (log)</th>"
         f"<th style='{_CS}background:#f1f5f9;'>p</th>"
         f"<th style='{_CS}background:#f1f5f9;'>Implicit hatás</th>"
         f"<th style='{_CS}background:#f1f5f9;'>95% CI (hatás)</th></tr>")
    rows = []
    for _, r in t.iterrows():
        bg = "background:#fef3c7;" if r['valtozo'] in highlight else ""
        rows.append(
            f"<tr><td style='{_CS}{bg}'><code>{r['valtozo']}</code></td>"
            f"<td style='{_CS}{bg}'>{fnum(r['coef'])}</td>"
            f"<td style='{_CS}{bg}'>{fnum(r['p'], 4)}</td>"
            f"<td style='{_CS}{bg}'>{fpct(r['hatas_pct'], 1)}</td>"
            f"<td style='{_CS}{bg}'>[{fpct(r['ci_lo_pct'], 1)}; {fpct(r['ci_hi_pct'], 1)}]</td></tr>")
    return f"<table style='border-collapse:collapse;font-size:12px;margin:6px 0 14px 0;'>{h}{''.join(rows)}</table>"



### 1. Nyers gradiens: sáv-mediánok, referencia-zóna és érzékenység

**— Módszer.** A vágánytól mért légvonalbeli távolság szerint sávokra osztjuk a pontos eladó mintát, és sávonként kiszámítjuk a fajlagos ár mediánját és átlagát. A diszkontot a **referencia-zónához** viszonyítjuk: ez a legtávolabbi sáv, amelyben legalább 20 megfigyelés van (adaptív referencia). A különbségek szignifikanciáját sávonként Mann–Whitney U (nemparaméteres) teszttel ellenőrizzük.

**— Miért "nyers"?** A sávmediánokban a lakások összetétele (méret, kor, állapot, épülettípus) eltér — a nyers mediánarány ezért **összetételi torzítást** tartalmaz, és csak kiindulópontnak tekinthető. A tiszta hatást a 2. lépés kontrollált modellje izolálja. Az érzékenységi tábla a nyers <150 m-es diszkontot minden jelölt referencia mellett megmutatja — a főmutató mindig az adaptív referencia sora.


In [ ]:

# --- 1. NYERS gradiens (a gyorstárból; itt nincs számítás, csak megjelenítés) ---
G = A.immission_gradient(AREA)
sv = G['savok'].copy()
sv = sv.rename(columns={sv.columns[0]: 'sav'})
ref = G['referencia'].iloc[0]
ref_zona = ref['ref_zona']

sv_disp = sv.copy()
sv_disp['median'] = sv_disp['median'].map(lambda v: fmt_huf(v))
sv_disp['atlag'] = sv_disp['atlag'].map(lambda v: fmt_huf(v))
display(HTML("<h4>Sávstatisztika: medián és átlag fajlagos ár sávonként (a referencia-sáv kiemelve)</h4>"))
display(HTML(plain_table_html(sv_disp, highlight_col='sav', highlight_val=ref_zona)))

fig1 = px.bar(sv, x='sav', y='median',
              labels={'sav': 'Immissziós sáv (vágánytól, m)', 'median': 'Medián ár/m<sup>2</sup> (Ft)'},
              template=PLOTLY_TEMPLATE)
fig1.data[0].marker.color = ['#2563eb' if s != ref_zona else '#94a3b8' for s in sv['sav']]
fig1.update_layout(showlegend=False, height=420, xaxis_tickangle=-25,
                   title=f"Medián fajlagos ár a vasúti immissziós sávok szerint — referencia: {ref_zona}")
fig1.show()

has_u150 = (sv['sav'] == '<150 m').any()
med_u150 = float(sv.loc[sv['sav'] == '<150 m', 'median'].iloc[0]) if has_u150 else np.nan
n_u150 = int(sv.loc[sv['sav'] == '<150 m', 'N'].iloc[0]) if has_u150 else 0
nyers_diskont = ((med_u150 - ref['med_ref']) / ref['med_ref'] * 100) if pd.notna(med_u150) and ref['med_ref'] > 0 else np.nan
display(HTML(kpi_grid_html([
    ("<150 m-es sáv (N)", f"{n_u150} db", "közvetlen immisszió", "#dc2626"),
    ("Medián ár <150 m", fmt_huf(med_u150), "fajlagos ár", "#dc2626"),
    (f"Referencia: {ref_zona} (N)", f"{int(ref['N_ref'])} db", "adaptív referencia-zóna", "#1e3a8a"),
    ("Medián ár — referencia", fmt_huf(ref['med_ref']), ref_zona, "#1e3a8a"),
    ("Nyers diszkont (<150 m)", fpct(nyers_diskont), "a referencia mediánjához képest — összetételi torzítással", "#f59e0b"),
])))

sens = G['erzekenyseg'].copy()
sens['nyers_diszkont_150_pct'] = sens['nyers_diszkont_150_pct'].map(lambda v: fpct(v, 1))
display(HTML("<h4>Referencia-érzékenység: a nyers <150 m-es diszkont minden jelölt referencia mellett</h4>"))
display(HTML(plain_table_html(sens, highlight_col='referencia', highlight_val=ref_zona)))

mwu = G['mwu'].copy()
mwu['U'] = mwu['U'].map(lambda v: f"{v:,.0f}".replace(',', ' '))
mwu['p'] = mwu['p'].map(lambda v: f"{v:.4f}".replace('.', ','))
display(HTML(f"<h4>Mann–Whitney U-tesztek: sávok vs. referencia ({ref_zona})</h4>"))
display(HTML(plain_table_html(mwu)))

p_u150 = float(G['mwu'].loc[G['mwu']['sav'] == '<150 m', 'p'].iloc[0]) if (G['mwu']['sav'] == '<150 m').any() else np.nan
p_u150_txt = f"{p_u150:.4f}".replace('.', ',') if pd.notna(p_u150) else "n.a."
print(f"A nyers mediánarány szerint a <150 m-es sáv {fpct(nyers_diskont, 1)} eltérést mutat "
      f"a(z) {ref_zona} referencia-zónához képest (Mann–Whitney U p = {p_u150_txt}). "
      "Ez a szám összetételi torzítást tartalmaz: a sávok lakásállománya eltér "
      "(méret, kor, állapot, épülettípus) — a tiszta, kontrollált hatást a 2. lépés hedonikus modellje adja meg.")



### 2. A kanonikus hedonikus modell: kontrollált zajdiszkont és TOD-prémium

**— Specifikáció.** A függő változó a fajlagos ár logaritmusa; a magyarázó változók a fizikai kontrollok (alapterület, szobaszám, panel, lift, állapot, erkély, emelet, épületkor), a kötöttpálya-TOD dummyk és az immissziós sáv-dummyk; a becslés robusztus (HC1) standard hibákkal. Két változatot hasonlítunk össze:

- **Zaj-fókuszú modell** (`model_zones`): fizikai kontrollok + sáv-dummyk — ebből olvassuk le a fejezet főmutatóját, a **zajdiszkontot**.
- **Teljes modell** (`model`): ugyanez kiegészítve a kötöttpálya-állomások gyalogos izokrón-dummyival — a kettő együtt mutatja a vasút kettős természetét. (A vasút-specifikus TOD-dummyk kimaradnak: az állomások a vágányoknál vannak, így a vasúti TOD és a zajsáv kollineáris lenne — ez a kanonikus specifikáció része.)

**— Miért a küszöbös (sáv-dummy) alak a fő specifikáció?** A küszöbös forma nem kényszerít simaságot: megengedi, hogy a diszkont a <150 m-es sávban ugrásszerűen, távolabb fokozatosan csengjen le, és együtthatói közvetlenül, sávonként értelmezhetők. A folytonos (log-vágánytáv) változat csak robusztusság-ellenőrzésre való; itt a robusztusságot a zaj-fókuszú és a teljes specifikáció összevetése képviseli.


In [ ]:

# --- 2. KANONIKUS hedonikus modell (együttható-táblák és KPI-k a gyorstárból) ---
H = A.canonical_hedonic(AREA)
meta = H['meta'].iloc[0]
zone_cols = [z.strip() for z in str(meta['zone_cols']).split(',')]
tod_cols = [t.strip() for t in str(meta['tod_cols']).split(',')]
ref_zona = meta['ref_zona']

zaj = H['zaj'].copy()
teljes = H['teljes'].copy()

zaj_disp = coef_pct(zaj)
display(HTML("<h4>Zaj-fókuszú modell (model_zones): fizikai kontrollok + immissziós sáv-dummyk "
             "(zóna-sorok kiemelve, implicit %-os hatással)</h4>"))
display(HTML(coef_table_html(zaj_disp, highlight=set(zone_cols))))

r150 = zaj[zaj['valtozo'] == '<150 m']
zajdiszkont = (np.exp(float(r150['coef'].iloc[0])) - 1) * 100 if len(r150) else np.nan
tod_col = tod_cols[0] if tod_cols else None
r_tod = teljes[teljes['valtozo'] == tod_col] if tod_col else teljes.iloc[0:0]
tod_pct = (np.exp(float(r_tod['coef'].iloc[0])) - 1) * 100 if len(r_tod) else np.nan
d_r2 = meta['r2_teljes'] - meta['r2_zaj']
display(HTML(kpi_grid_html([
    ("Becslési minta (N)", f"{int(meta['n'])} db", "pontos eladó, teljes modell", "#1e3a8a"),
    ("R<sup>2</sup> — zaj-fókuszú", fnum(meta['r2_zaj']), "fizikai kontrollok + sávok", "#2563eb"),
    ("R<sup>2</sup> — teljes", fnum(meta['r2_teljes']), "fizikai + TOD + sávok", "#2563eb"),
    ("Zajdiszkont (<150 m)", fpct(zajdiszkont), f"kontrollált, referencia: {ref_zona}", "#dc2626"),
    (f"TOD-prémium ({tod_col})", fpct(tod_pct), "5 perces kötöttpálya-séta (0-375 m)", "#059669"),
])))

teljes_disp = coef_pct(teljes)
display(HTML("<h4>Teljes modell: fizikai kontrollok + kötöttpálya TOD-dummyk + sávok</h4>"))
display(HTML(coef_table_html(teljes_disp, highlight=set(zone_cols) | set(tod_cols))))

z_sorted = zaj_disp.sort_values('coef')
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=z_sorted['hatas_pct'], y=z_sorted['valtozo'], mode='markers',
    marker=dict(size=10, color=['#dc2626' if v in zone_cols else '#2563eb' for v in z_sorted['valtozo']]),
    error_x=dict(type='data', symmetric=False,
                 array=(z_sorted['ci_hi_pct'] - z_sorted['hatas_pct']).values,
                 arrayminus=(z_sorted['hatas_pct'] - z_sorted['ci_lo_pct']).values),
    customdata=z_sorted['p'].values,
    hovertemplate='%{y}: %{x:.1f}% (p=%{customdata:.4f})<extra></extra>'))
fig2.add_vline(x=0, line_dash='dash', line_color='#475569')
fig2.update_layout(template=PLOTLY_TEMPLATE, height=460,
    title='Zaj-fókuszú modell: implicit hatások 95% CI-vel (zóna-sorok pirossal)',
    xaxis_title='Implicit hatás a fajlagos árra (%)')
fig2.show()

r150_t = teljes[teljes['valtozo'] == '<150 m']
zajdiszkont_t = (np.exp(float(r150_t['coef'].iloc[0])) - 1) * 100 if len(r150_t) else np.nan
print(f"A két specifikáció összevetése: a kötöttpálya TOD-dummyk hozzáadása "
      f"dR<sup>2</sup> = {fnum(d_r2, 4)}; a <150 m-es zajdiszkont {fpct(zajdiszkont, 1)} (zaj) vs. "
      f"{fpct(zajdiszkont_t, 1)} (teljes) — a sáv-együtthatók lényegileg stabilak, vagyis "
      "a környezeti (zaj) és az elérhetőségi (TOD) hatás külön csatornán fut. "
      "A folytonos (log-vágánytáv) robusztusság-modell szándékosan kimarad: "
      "a kánon szerint a küszöbös (sáv-dummy) a főspecifikáció, a robusztusságot "
      "ez a két-specifikációs összevetés képviseli.")



### 3. Dózis-válasz, placebo-küszöb és strukturális törés (Chow)

**— Kérdés:** tényleg sávonként cseng le a diszkont, és hol a hatás töréspontja?

- **Dózis-válasz:** a zaj-modell sáv-együtthatóit a sávok középtávolságán ábrázoljuk — ha az implicit diszkont a távolsággal fokozatosan szűnik meg, a görbe lecsengő mintázatot mutat.
- **Placebo/Chow:** a ~300 m-es törést formális tesztnek vetjük alá: a kontrollált modellt a <300 m és a ≥300 m almintára külön-külön is becsüljük, és Chow-próbával ellenőrizzük, hogy a paraméterek azonosak-e a két oldalon. Ha a próba elutasít, a vágányközeli szegmens árazési szerkezete statisztikailag különbözik — a sávhatár nem pusztán konvenció.


In [ ]:

# --- 3. Dózis-válasz + placebo/Chow (a gyorstárból) ---
chow = G['chow'].iloc[0]
chow_disp = G['chow'].copy()
chow_disp['f'] = chow_disp['f'].map(lambda v: fnum(v, 2))
chow_disp['p'] = chow_disp['p'].map(lambda v: f"{v:.4f}".replace('.', ','))
display(HTML("<h4>Chow-teszt a 300 m-es törésre (azonosak-e a paraméterek a két oldalon?)</h4>"))
display(HTML(plain_table_html(chow_disp)))

p_chow_txt = f"{chow['p']:.4f}".replace('.', ',')
chow_verdict = ("elutasítja a paraméter-azonosságot: a <300 m és a >=300 m szegmens árazési "
                "szerkezete statisztikailag különbözik"
                if chow['p'] < 0.05
                else "nem utasítja el a paraméter-azonosságot: a 300 m-es határ nem bizonyul "
                "szignifikáns törésnek")
print(f"Chow-teszt (300 m): F = {fnum(chow['f'], 2)}, p = {p_chow_txt} "
      f"(k = {int(chow['k'])}, N = {int(chow['n_full'])} = {int(chow['n_a'])} + {int(chow['n_b'])}) "
      f"— a teszt {chow_verdict}.")

# lecsengő mintázat a zaj-modell zóna-együtthatóiból, sávközépponti távolságokon
ZONE_MID = {'<150 m': 75.0, '150-300 m': 225.0, '300-500 m': 400.0,
            '500-1000 m': 750.0, '1000-2000 m': 1500.0, '>2000 m (referencia)': 3000.0}
decay = zaj_disp[zaj_disp['valtozo'].isin(zone_cols)][['valtozo', 'hatas_pct', 'ci_lo_pct', 'ci_hi_pct']].copy()
decay['tav'] = decay['valtozo'].map(ZONE_MID)
ref_row = pd.DataFrame([{'valtozo': f"{ref_zona} (referencia)", 'hatas_pct': 0.0,
                         'ci_lo_pct': 0.0, 'ci_hi_pct': 0.0,
                         'tav': ZONE_MID.get(ref_zona, np.nan)}])
decay = pd.concat([decay, ref_row], ignore_index=True).sort_values('tav')
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=decay['tav'], y=decay['hatas_pct'], mode='lines+markers',
    marker=dict(size=10, color='#dc2626'), line=dict(color='#dc2626', width=2.5),
    error_y=dict(type='data', symmetric=False,
                 array=(decay['ci_hi_pct'] - decay['hatas_pct']).values,
                 arrayminus=(decay['hatas_pct'] - decay['ci_lo_pct']).values,
                 thickness=1.5, color='#dc2626'),
    text=decay['valtozo'], hovertemplate='%{text}: %{y:.1f}%<extra></extra>'))
fig3.add_hline(y=0, line_dash='dash', line_color='#475569')
fig3.add_vline(x=300, line_dash='dot', line_color='#7c3aed', annotation_text='Chow-határ (300 m)')
fig3.update_layout(template=PLOTLY_TEMPLATE, height=420,
    title='Dózis-válasz: a zajdiszkont lecsengése a vágánytól mért távolsággal (kontrollált sáv-együtthatók, %)',
    xaxis_title='Távolság a vágánytól (m, sávközéppont)', yaxis_title='Implicit diszkont (%)')
fig3.show()



### 4. Térbeli függés: globális Moran-féle I és a reziduum-térkép

**— Miért fontos?** Az ingatlanárak térben nem függetlenek: a közeli lakások árai együttmozognak (Tobler első földrajzi törvénye). Ha a kontrollált modell hibatagjai térben korrelálnak, az OLS becslések és a tesztek érvényessége sérül — erre a 8 legközelebbi szomszéd (KNN-8) súlymátrixszal, a reziduumokon számított **globális Moran I** a formális próba, permutációs p-értékkel.

**— A reziduum-térkép** azt mutatja, hol becsül a modell alul (pozitív reziduum: a hirdetés olcsóbb a modell áránál) és felül (negatív reziduum) — a klaszteres mintázat a kihagyott térbeli tényezők nyoma.


In [ ]:

# --- 4. Moran-féle I + reziduum-térkép (a gyorstárból) ---
MA = A.moran_analysis(AREA)
mor = MA['moran'].iloc[0]
p_mor_txt = f"{mor['p_perm']:.3f}".replace('.', ',')
display(HTML(kpi_grid_html([
    ("Moran I (reziduumok)", fnum(mor['I'], 4), f"várható érték: {fnum(mor['EI'], 4)}", "#7c3aed"),
    ("Permutációs p", p_mor_txt, "999 permutáció, KNN-8", "#7c3aed"),
    ("z-érték", fnum(mor['z'], 2), "standardizált statisztika", "#7c3aed"),
])))
mor_txt = ("szignifikáns térbeli autokorreláció a hibatagokban: az OLS térbeli "
           "függetlenségi feltevése sérül, térökonometriai korrekció indokolt"
           if mor['p_perm'] < 0.05
           else "nem igazolható térbeli autokorreláció a hibatagokban")
print(f"Moran I = {fnum(mor['I'], 4)} (E[I] = {fnum(mor['EI'], 4)}, z = {fnum(mor['z'], 2)}, "
      f"permutációs p = {p_mor_txt}) → {mor_txt}.")

# reziduumok térképen — ugyanaz a szűrés, mint az analyses.moran_analysis bemenete
d_m = p.dropna(subset=["log_nm_ar"] + [c for c in CANON_PHYSICAL if c in p.columns]).copy()
res = MA['ols_resid'].copy()
if len(res) == len(d_m):
    res['geokodolt_lon'] = d_m['geokodolt_lon'].values
    res['geokodolt_lat'] = d_m['geokodolt_lat'].values
    res_map = res.dropna(subset=['geokodolt_lon', 'geokodolt_lat'])
    fig4 = px.scatter_map(res_map, lat='geokodolt_lat', lon='geokodolt_lon',
                          color='resid', color_continuous_scale='RdBu_r',
                          color_continuous_midpoint=0,
                          opacity=0.85, template=PLOTLY_TEMPLATE, height=520,
                          labels={'resid': 'Reziduum (log)'})
    fig4.update_traces(marker=dict(size=7))
    fig4.update_layout(margin=dict(l=10, r=10, t=60, b=10),
                       title='A kontrollált OLS reziduumai térben (kék: alulárazott, piros: túlárazott pontok)')
    fig4.show()
else:
    print("A reziduum-térkép nem készült el: a reziduumok száma nem illeszkedik a szűrt mintához.")



### 5. Térökonometria: SAR és SEM

**— Két vetélytárs modell.** Ha a térbeli függés az árakon keresztül hat (a szomszédos ingatlanok ára közvetlenül befolyásolja egymást — presztízs- és referenciahatás), a **SAR** modell a megoldás; ha a függés a ki nem mért tényezőkön (mikro-lokáció, utcakép, közbiztonség) keresztül a hibatagba szivárog, a **SEM**. A SAR legfontosabb hozadéka a **térbeli multiplikátor** (1/(1-ρ)): egy helyi árnövekmány a szomszédsági láncon keresztül ennyiszeres összhatást generál. A specifikációválasztást hagyományosan az LM-tesztek (LM-lag, LM-error és robusztus változataik) segítik; ha ezek nem állnak rendelkezésre, a két modell illeszkedését és a zajsáv-együtthatók stabilitását vetjük össze.


In [ ]:

# --- 5. Térökonometria: SAR és SEM (a gyorstárból) ---
SS = A.sar_sem(AREA)
zone_cols = [z.strip() for z in str(meta['zone_cols']).split(',')]

lm = SS['lm']
lm_ok = bool(lm['stat'].notna().all())
if lm_ok:
    lm_disp = lm.copy()
    lm_disp['stat'] = lm_disp['stat'].map(lambda v: fnum(v, 3))
    lm_disp['p'] = lm_disp['p'].map(lambda v: fnum(v, 4))
    display(HTML("<h4>LM specifikáció-tesztek (Anselin)</h4>"))
    display(HTML(plain_table_html(lm_disp)))
else:
    print("Az LM specifikáció-tesztek nem állnak rendelkezésre (a spreg-diagnosztika ebben a "
          "környezetben nem futott le; NaN értékek). A specifikációválasztást ezért a "
          "Moran-tesztre, valamint a SAR és SEM becslések közvetlen összevetésére alapozzuk "
          "— ezt őszinten jelezzük.")

sar_ok = len(SS.get('sar', pd.DataFrame())) > 0 and 'sar_hiba' not in SS
sem_ok = len(SS.get('sem', pd.DataFrame())) > 0 and 'sem_hiba' not in SS
if 'sar_hiba' in SS:
    print(f"A SAR modell nem futott le: {SS['sar_hiba']['hiba'].iloc[0]}")
if 'sem_hiba' in SS:
    print(f"A SEM modell nem futott le: {SS['sem_hiba']['hiba'].iloc[0]}")

kpi5 = []
rho_row = SS['sar_rho'].iloc[0] if sar_ok else None
lam_row = SS['sem_lambda'].iloc[0] if sem_ok else None
if rho_row is not None:
    kpi5 += [("SAR ρ", fnum(rho_row['rho']), "térbeli autoregressziós paraméter", "#7c3aed"),
             ("Térbeli multiplikátor", f"{rho_row['mult']:.3f}".replace('.', ','),
              "1/(1-ρ) — tovagyűrűző összhatás", "#7c3aed")]
if lam_row is not None:
    kpi5.append(("SEM λ", fnum(lam_row['lambda']), "térbeli hibakorreláció", "#2563eb"))
display(HTML(kpi_grid_html(kpi5)))

mult_txt = f"{rho_row['mult']:.3f}".replace('.', ',') if rho_row is not None else "n.a."
if rho_row is not None:
    print(f"A SAR modellben ρ = {fnum(rho_row['rho'])}: a becsült térbeli multiplikátor "
          f"1/(1-ρ) = {mult_txt} — egy helyi árnövekmány a szomszédsági láncon "
          "keresztül ennyiszeres összhatást generál.")
if lam_row is not None:
    print(f"A SEM modellben λ = {fnum(lam_row['lambda'])} — a hibatagok térbeli "
          "korrelációjának erőssége.")

# együttható-összehasonlítés a zajzóna-sorokra: OLS vs. SAR vs. SEM
cmp = SS['ols'][['valtozo', 'coef']].rename(columns={'coef': 'coef_ols'}).copy()
cmp = cmp[cmp['valtozo'].isin(zone_cols)].copy()
if sar_ok:
    sar_t = SS['sar'][SS['sar']['valtozo'].isin(zone_cols)][['valtozo', 'coef']].rename(columns={'coef': 'coef_sar'})
    cmp = cmp.merge(sar_t, on='valtozo', how='left')
if sem_ok:
    sem_t = SS['sem'][SS['sem']['valtozo'].isin(zone_cols)][['valtozo', 'coef']].rename(columns={'coef': 'coef_sem'})
    cmp = cmp.merge(sem_t, on='valtozo', how='left')
for c in [c for c in ['coef_ols', 'coef_sar', 'coef_sem'] if c in cmp.columns]:
    cmp[c] = (np.exp(cmp[c]) - 1) * 100
ren = {'valtozo': 'zajsáv'}
if 'coef_ols' in cmp.columns: ren['coef_ols'] = 'OLS (%)'
if 'coef_sar' in cmp.columns: ren['coef_sar'] = 'SAR (%)'
if 'coef_sem' in cmp.columns: ren['coef_sem'] = 'SEM (%)'
cmp_disp = cmp.rename(columns=ren).copy()
for c in [c for c in cmp_disp.columns if c.endswith('(%)')]:
    cmp_disp[c] = cmp_disp[c].map(lambda v: fpct(v, 1))
display(HTML(f"<h4>Zajsáv-együtthatók: OLS vs. SAR vs. SEM (implicit %, referencia: {ref_zona})</h4>"))
display(HTML(plain_table_html(cmp_disp)))



### 6. Lokális térökonometria: GWR — hol erős a vasúti hatás?

**— Módszer.** A GWR (geographically weighted regression) minden megfigyelésre saját, szomszédsággal súlyozott regressziót illeszt, így a vasúti zaj együtthatója (a log-vágánytáv meredeksége) pontról pontra változik. A térkép a **log_vasut_m együtthatójának β-felületét** mutatja, szignifikancia-maszkolva (|t| ≥ 1,96).

**— Épületszintű / jitterelt minta.** Az azonos koordinátájú (azonos épületben lévő) hirdetéseket a lokális ablakok szingularitásának elkerülésére a modul automatikusan kezeli: nagy mintánál épületszintre aggregál (kerekített koordináták átlagolása), kis mintánál jittereli a duplikált pontokat (seedelt, +/-1,5e-5 fok). Az alkalmazott változatot a kód kiírja. Ha a GWR illesztés metaadatai (sávszélesség, R<sup>2</sup>, AICc) nem állnak rendelkezésre, azokat őszinten kihagyjuk.


In [ ]:

# --- 6. GWR (a gyorstárból; ha a meta hiányzik, őszinte kihagyás) ---
GW = A.gwr_analysis(AREA)
modszer = GW['modszer'].iloc[0]
print(f"GWR módszer: {modszer['modszer']} — magyarázó változók: {modszer['feat']}")

hiba_txt = GW['gwr_hiba']['hiba'].iloc[0] if ('gwr_hiba' in GW and len(GW['gwr_hiba']) > 0) else None
meta_g = GW.get('meta_gwr')
meta_ok = (hiba_txt is None) and meta_g is not None and len(meta_g) > 0 and pd.notna(meta_g['bw'].iloc[0])
params_ok = ('params' in GW and 'tvals' in GW and len(GW['params']) > 0
             and 'log_vasut_m' in GW['params'].columns)

if hiba_txt is not None:
    print(f"A GWR illesztés metaadatai nem érhetők el — őszinte kihagyás, indok: {hiba_txt}")

if meta_ok:
    mg = meta_g.iloc[0]
    sig_n = int(mg['sig_zaj']) if pd.notna(mg.get('sig_zaj', np.nan)) else int((GW['tvals']['log_vasut_m'].abs() >= 1.96).sum())
    display(HTML(kpi_grid_html([
        ("GWR sávszélesség (bw)", f"{int(mg['bw'])}", "adaptív lokális ablak", "#7c3aed"),
        ("GWR R<sup>2</sup>", fnum(mg['r2']), f"adj. R<sup>2</sup> = {fnum(mg['adj_r2'])}", "#2563eb"),
        ("GWR AICc", fnum(mg['aicc'], 1), "modellválasztási kritérium", "#2563eb"),
        ("Szignifikáns pontok", f"{sig_n} / {len(GW['params'])}", "|t| >= 1,96 a log_vasut_m-re", "#059669"),
    ])))

if params_ok:
    b = GW['params'][['geokodolt_lon', 'geokodolt_lat', 'log_vasut_m']].copy()
    t = GW['tvals']['log_vasut_m'].values
    sig_n = int((np.abs(t) >= 1.96).sum())
    b['t'] = t
    b['b_sig'] = np.where(np.abs(t) >= 1.96, b['log_vasut_m'], np.nan)
    rng = float(b['log_vasut_m'].abs().max())
    if not np.isfinite(rng) or rng == 0:
        rng = 1.0
    fig6 = px.scatter_map(b, lat='geokodolt_lat', lon='geokodolt_lon',
                          color='log_vasut_m', color_continuous_scale='RdBu_r',
                          color_continuous_midpoint=0, range_color=[-rng, rng],
                          opacity=0.4, template=PLOTLY_TEMPLATE, height=520,
                          labels={'log_vasut_m': 'β (log_vasut_m)'})
    sig = b.dropna(subset=['b_sig'])
    if len(sig) > 0:
        fig6.add_trace(px.scatter_map(sig, lat='geokodolt_lat', lon='geokodolt_lon',
                                      color='b_sig', color_continuous_scale='RdBu_r',
                                      color_continuous_midpoint=0, range_color=[-rng, rng],
                                      opacity=0.95, template=PLOTLY_TEMPLATE,
                                      labels={'b_sig': 'β (log_vasut_m)'}).data[0])
        fig6.data[-1].update(marker=dict(showscale=False))
    fig6.update_traces(marker=dict(size=8))
    fig6.update_layout(margin=dict(l=10, r=10, t=70, b=10),
                       title=f"GWR: a log-vágánytáv lokális együtthatója "
                             f"(szignifikancia-maszkolt, |t| >= 1,96; {sig_n}/{len(b)} pont) "
                             f"— módszer: {modszer['modszer']}")
    fig6.show()
    if hiba_txt is not None:
        print(f"Megjegyzés: a β-felület a meglévő paraméter-táblákból jelenik meg, "
              f"de a GWR-meta (bw, R<sup>2</sup>, AICc) a fenti hiba miatt kimaradt: {hiba_txt}")
else:
    print("A GWR paraméter-felületek nem érhetők el — a β-térkép őszinten kimarad.")



## Zárás: a hat lépés objektív eredményei

Az alábbi cella a fejezet számait a gyorstárból, f-stringgel foglalja mondatokba — nincs benne kézzel beírt érték. A hat lépés együtt a vasúti árhatás teljes mérési láncát adja: nyers → kontrollált → térbeli.


In [ ]:

# --- Záró összefoglaló: minden szám a gyorstárból, f-stringgel ---
chow = G['chow'].iloc[0]
mor = MA['moran'].iloc[0]
ref = G['referencia'].iloc[0]
sv2 = G['savok'].copy()
sv2 = sv2.rename(columns={sv2.columns[0]: 'sav'})
med_u150 = float(sv2.loc[sv2['sav'] == '<150 m', 'median'].iloc[0]) if (sv2['sav'] == '<150 m').any() else np.nan
nyers_diskont = ((med_u150 - ref['med_ref']) / ref['med_ref'] * 100) if pd.notna(med_u150) and ref['med_ref'] > 0 else np.nan
p_u150 = float(G['mwu'].loc[G['mwu']['sav'] == '<150 m', 'p'].iloc[0]) if (G['mwu']['sav'] == '<150 m').any() else np.nan
p_u150_txt = f"{p_u150:.4f}".replace('.', ',') if pd.notna(p_u150) else "n.a."
zajdiszkont = (np.exp(float(H['zaj'].loc[H['zaj']['valtozo'] == '<150 m', 'coef'].iloc[0])) - 1) * 100 \
    if (H['zaj']['valtozo'] == '<150 m').any() else np.nan
tod_pct = (np.exp(float(H['teljes'].loc[H['teljes']['valtozo'] == tod_col, 'coef'].iloc[0])) - 1) * 100 \
    if (tod_col and (H['teljes']['valtozo'] == tod_col).any()) else np.nan
p_chow_txt = f"{chow['p']:.4f}".replace('.', ',')
p_mor_txt = f"{mor['p_perm']:.3f}".replace('.', ',')
mult_txt = f"{SS['sar_rho']['mult'].iloc[0]:.3f}".replace('.', ',') if ('sar_rho' in SS and len(SS['sar_rho']) > 0) else "n.a."
rho_txt = fnum(SS['sar_rho']['rho'].iloc[0]) if ('sar_rho' in SS and len(SS['sar_rho']) > 0) else "n.a."
lam_txt = fnum(SS['sem_lambda']['lambda'].iloc[0]) if ('sem_lambda' in SS and len(SS['sem_lambda']) > 0) else "n.a."

print(f"1) Nyers gradiens ({area_nev}): a <150 m-es sáv medián fajlagos ára {fmt_huf(med_u150)}, "
      f"a(z) {ref['ref_zona']} referencia-zóna mediánja {fmt_huf(ref['med_ref'])} — a nyers "
      f"mediánarány {fpct(nyers_diskont, 1)} diszkontot mutat (Mann–Whitney U p = {p_u150_txt}), "
      "de összetételi torzítást tartalmaz.")
print(f"2) Kontrollált modell: a zaj-fókuszú specifikáció szerint a <150 m-es sáv implicit hatása "
      f"{fpct(zajdiszkont, 1)} a(z) {meta['ref_zona']} referencia-zónához képest "
      f"(R<sup>2</sup>_zaj = {fnum(meta['r2_zaj'])}, R<sup>2</sup>_teljes = {fnum(meta['r2_teljes'])}, "
      f"N = {int(meta['n'])}); a TOD-prémium {fpct(tod_pct, 1)} ({tod_col}, 5 perces kötöttpálya-séta).")
print(f"3) Dózis-válasz és törés: a Chow-teszt a 300 m-es határra F = {fnum(chow['f'], 2)}, "
      f"p = {p_chow_txt} — "
      + ("a két oldal árazési szerkezete statisztikailag különbözik."
         if chow['p'] < 0.05 else "a 300 m-es határ nem bizonyul szignifikáns törésnek."))
print(f"4) Térbeli függés: Moran I = {fnum(mor['I'], 4)} a kontrollált OLS reziduumain "
      f"(permutációs p = {p_mor_txt}) — "
      + ("szignifikáns térbeli autokorreláció, térökonometriai korrekció indokolt."
         if mor['p_perm'] < 0.05 else "nincs igazolható térbeli autokorreláció."))
print(f"5) Térökonometria: SAR ρ = {rho_txt} (térbeli multiplikátor: {mult_txt}), "
      f"SEM λ = {lam_txt}; a zajsáv-együtthatók OLS/SAR/SEM közötti eltérése "
      "a fenti összehasonlító táblában látható.")
gwr_meta_txt = "n.a."
if 'meta_gwr' in GW and len(GW['meta_gwr']) > 0 and pd.notna(GW['meta_gwr']['bw'].iloc[0]):
    mg = GW['meta_gwr'].iloc[0]
    gwr_meta_txt = (f"bw = {int(mg['bw'])}, R<sup>2</sup> = {fnum(mg['r2'])}, AICc = {fnum(mg['aicc'], 1)}, "
                    f"szignifikáns pontok = {int(mg['sig_zaj'])}")
print(f"6) GWR ({modszer['modszer']}): a log-vágánytáv lokális együtthatója térben változik; "
      f"meta: {gwr_meta_txt}."
      + (" A GWR-meta a fenti hiba miatt őszinten kimaradt." if hiba_txt is not None else ""))
